# Fetch TCEQ NSR Air-Permit Data

Source: https://www2.tceq.texas.gov/airperm/index.cfm?fuseaction=airpermits.start

For each region and permit type, submit the TCEQ search form, parse the returned
HTML table, and save one CSV per combination to:

```
data/raw/air-permits/{std,psd,ghgpsd}/air-permit-{region}-{type}.csv
```

In [1]:
import time
from io import StringIO
from pathlib import Path

from lxml import html
import pandas as pd
import requests

ENDPOINT = 'https://www2.tceq.texas.gov/airperm/index.cfm'
FORM_URL = ENDPOINT + '?fuseaction=airpermits.start'

# Create project root
def find_project_root(marker: str = '.git') -> Path:
    '''Climbs up folders starting from CWD until it finds the root.'''
    current = Path.cwd().resolve()
    
    # Loop upward until hitting the project root (where current == its own parent)
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent

PROJECT_ROOT = find_project_root(marker='.git')
OUT_DIR = PROJECT_ROOT / 'data' / 'raw' / 'air-permits'

## Values each form field accepts

In [2]:
def discover_fields(form_url: str = FORM_URL, preview: int = 4) -> dict:
    headers = {'User-Agent': ''}
    doc = html.fromstring(requests.get(form_url, headers=headers, timeout=60).text)

    dropdowns = {}
    # Dropdowns: accepted values are <option value='...'>
    for select in doc.xpath('//select'):
        dropdowns[select.get('name')] = [
            (o.get('value'), (o.text or '').strip()) for o in select.xpath('.//option')
        ]

    # Print dropdown preview
    print('------------- Dropdown -------------')
    for name, options in dropdowns.items():
        print(f'\n# {name}  ({len(options)} values)')
        for val, label in options[:preview]:
            print(f'    {val!r:14} {label}')
        if len(options) > preview:
            print(f'    ... (+{len(options) - preview} more)')

    radio_or_checkboxes = {}
    # Radio buttons / checkboxes: each input's value is an accepted choice
    for input in doc.xpath("//input[@type='radio' or @type='checkbox']"):
        radio_or_checkboxes.setdefault(input.get('name'), [])
        radio_or_checkboxes[input.get('name')].append((input.get('value'), ''))


    # Print radios / checkboxes preview
    print('------------- Radio/Checkboxes -------------')
    for name, options in radio_or_checkboxes.items():
        print(f'\n# {name}  ({len(options)} values)')
        for val, label in options[:preview]:
            print(f'    {val!r:14} {label}')
        if len(options) > preview:
            print(f'    ... (+{len(options) - preview} more)')

    return dropdowns, radio_or_checkboxes


dropdown_values, radio_or_checkboxes_values = discover_fields()

------------- Dropdown -------------

# loc_cnty_name  (255 values)
    '0'            Any County
    'ANDERSON'     ANDERSON
    'ANDREWS'      ANDREWS
    'ANGELINA'     ANGELINA
    ... (+251 more)

# tnrcc_region_cd  (18 values)
    '0'            Any Region
    '1'            REGION 01 - AMARILLO
    '2'            REGION 02 - LUBBOCK
    '3'            REGION 03 - ABILENE
    ... (+14 more)

# addn_id_typ_txt  (21 values)
    ''             Any Type
    'AMOC'         AMOC
    'CONSTOPPMT'   CONSTRUCTION OPERATING PERMIT
    'CONSTRUCT'    CONSTRUCTION
    ... (+17 more)

# proj_typ_txt  (26 values)
    ''             Any Type
    'AMEND'        AMENDMENT OR MODIFICATION
    'CERTIFICAT'   CERTIFICATION OF UN-REGISTERED UNITS
    'CERTPRMT'     CERTIFICATION OF PERMITTED OR REGISTERED UNITS
    ... (+22 more)

# unit_id  (146 values)
    '0'            None
    '5113'         ACI
    '15210'        ACID STORAGE, PUMPING AND DISPENSING
    '5026'         AEROSPACE EQUIPMENT AND PA

## Fetch files

In [3]:
def fetch(region: int, permit_type: str, status: str, date_info: list, session: requests.Session, units:list = ['0']) -> pd.DataFrame:
    request = {
        'fuseaction': 'airpermits.validate_search_criteria',
        'program': 'NSR',
        'out_form': 'web',          
        'tnrcc_region_cd': str(region),
        'addn_id_typ_txt': permit_type,
        'loc_cnty_name': '',        
        'proj_status_txt': status,
        'date_option': date_info[0],
        'date_range_from': date_info[1],
        'date_range_to': date_info[2],
        'unit_id': units
    }
    resp = session.post(ENDPOINT, data=request, timeout=120)
    resp.raise_for_status()

    # Using pandas instead of polars because polars doesn't have read_html.
    # Table 0 is a (1, 1) summary cell; table 1 is the results. When a search
    # matches no permits the results table is absent, so return empty instead.
    tables = pd.read_html(StringIO(resp.text))
    if len(tables) < 2:
        return pd.DataFrame()

    return tables[1]

## Configure & run

Regions: `6` = Region 06 (El Paso), `7` = Region 07 (Midland). 

Permit types: `STD` = Standard permit, `PSD` = Prevention of Significant Deterioration Permit, `GHGPSD` = Greenhouse Gas PSD

UNITS: `8340` = ELECTRIC GENERATING FACILITIES, `15627` = NATURAL GAS ELECTRIC GENERATING UNITS

Date option: [Date option, start date, end date]

In [4]:
# Change values here for desired requirements
REGIONS = [6, 7]
PERMIT_TYPES = [
    # (Request value, Folder name)
    ('STDPMT', 'std'),         
    ('PSD', 'psd'),           
    ('GHGPSD', 'ghgpsd')       
]
UNITS = ['8340', '15627']
STATUS_FILTER = 'ALL'
DATE_INFO = ['rcv_dt', '01/01/2023', '06/15/2026'] 
SLEEP_SECONDS = 1.0

session = requests.Session()
session.headers['User-Agent'] = ''

for region in REGIONS:
    for ptype, folder_name in PERMIT_TYPES:
        # Create destination folder
        dest = OUT_DIR / folder_name / f'air-permit-{region}.csv'
        dest.parent.mkdir(parents=True, exist_ok=True)

        # Download files
        if ptype == 'STDPMT':
            df = fetch(region, ptype, STATUS_FILTER, DATE_INFO, session, UNITS)
        else:
            df = fetch(region, ptype, STATUS_FILTER, DATE_INFO, session)
        df.to_csv(dest, index=False)
        
        print(f'Region {region:>2} {ptype:<7} -> {len(df):>6} rows  {dest.relative_to(PROJECT_ROOT)}')
        time.sleep(SLEEP_SECONDS)

Region  6 STDPMT  ->     12 rows  data/raw/air-permits/std/air-permit-6.csv
Region  6 PSD     ->      1 rows  data/raw/air-permits/psd/air-permit-6.csv
Region  6 GHGPSD  ->      1 rows  data/raw/air-permits/ghgpsd/air-permit-6.csv
Region  7 STDPMT  ->     97 rows  data/raw/air-permits/std/air-permit-7.csv
Region  7 PSD     ->     22 rows  data/raw/air-permits/psd/air-permit-7.csv
Region  7 GHGPSD  ->     12 rows  data/raw/air-permits/ghgpsd/air-permit-7.csv
